# Evaluate sleep fragmentation

### Import recordings

Load packages

In [1]:
from scipy import signal
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import os
from IPython.display import display
from ipyfilechooser import FileChooser
from scipy.stats import zscore
import json
import matplotlib.cm as cm
import IPython
import ast
from scipy.interpolate import interp1d
import pickle
from datetime import datetime
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta
import matplotlib.dates as mdates


Load functions

In [2]:
def flip_headstage(LFPs_df, ID):
    rec_ch_list_ID = LFPs_df.columns-ID*32
    rec_ch_list_mouse = [value for value in rec_ch_list_ID if 0 <= value <= 31]
    i = np.argmax(rec_ch_list_ID>=0)
    inverted_chs = np.concatenate([range(16,32,1), range(0,16,1)], axis=0)
    LFPs_df_mouse=LFPs_df.iloc[:,i:i+len(rec_ch_list_mouse)]
    flipped_ch=(inverted_chs[LFPs_df_mouse.columns-(ID*32)])+(ID*32)
    LFPs_df.columns.values[i:i+len(rec_ch_list_mouse)] = flipped_ch
    LFPs_df = LFPs_df.sort_index(axis=1)
    return LFPs_df

def regrouper_labels_consecutifs(df):
    """Regroupe les labels identiques qui se suivent"""
    episodes = []    
    if len(df) == 0:
        return pd.DataFrame()    
    # Initialiser le premier épisode
    current_label = df.iloc[0]['label']
    start_time = df.iloc[0]['time']
    total_duration = df.iloc[0]['duration']    
    for i in range(1, len(df)):
        if df.iloc[i]['label'] == current_label:
            # Même label, on accumule la durée
            total_duration += df.iloc[i]['duration']
        else:
            # Changement de label, on sauvegarde l'épisode
            episodes.append({
                'start_time': start_time,
                'end_time': df.iloc[i-1]['time'] + df.iloc[i-1]['duration'],
                'label': current_label,
                'duration': total_duration
            })            
            # Nouveau épisode
            current_label = df.iloc[i]['label']
            start_time = df.iloc[i]['time']
            total_duration = df.iloc[i]['duration']    
    # Ajouter le dernier épisode
    episodes.append({
        'start_time': start_time,
        'end_time': df.iloc[-1]['time'] + df.iloc[-1]['duration'],
        'label': current_label,
        'duration': total_duration
    })    
    return pd.DataFrame(episodes)

def closest_checkpoint(timestr: str) -> str:
    # Parse HH:MM:SS into a datetime (date doesn't matter)
    now = datetime.strptime(timestr, "%H:%M:%S")
    # Candidate times for *today*
    t7 = now.replace(hour=7, minute=0, second=0)
    t19 = now.replace(hour=19, minute=0, second=0)
    # If times already passed, add one day
    if t7 <= now:
        t7 += timedelta(days=1)
    if t19 <= now:
        t19 += timedelta(days=1)
    # Compute time differences
    diff7 = t7 - now
    diff19 = t19 - now
    # Return whichever is closer
    HalfDay="Night" if diff7 < diff19 else "Day"
    delaybeforeHalfDay=diff7 if diff7 < diff19 else diff19
    return HalfDay, delaybeforeHalfDay.total_seconds()

Process

In [ ]:
folder_path = "//10.69.168.1/crnldata/forgetting/Carla/OpenEphysData/Sleep/"

Summary_table=pd.DataFrame()
Summary_table2=pd.DataFrame()
EMGHypno_dict={}
counter=0
counter2=0

#Load LFP coordinates 
notebook_path = Path("/".join(IPython.extract_module_locals()[1]["__vsc_ipynb_file__"].split("/")[-5:]))
Channels = f'{notebook_path.parent}/_LFP_coordinates_of_all_mice.csv'
all_LFPcoordinates = pd.read_csv(Channels, index_col=0)
all_LFPcoordinates = pd.read_csv('_LFP_coordinates_of_all_mice.csv', sep=';', index_col=0)

#Load Implant ID
notebook_path = Path("/".join(IPython.extract_module_locals()[1]["__vsc_ipynb_file__"].split("/")[-5:]))
MouseID = f'{notebook_path.parent}/_MouseID_ImplantColorCode.csv'
all_MouseID = pd.read_csv(MouseID, sep=';', index_col=0)


print("Starts...")
for root, dirs, files in os.walk(folder_path):
    datfiles = [f for f in files if f.endswith(('.dat'))]
    for datfile in datfiles:
        #Path(root).parents[6].name
        datfilepath = os.path.join(root, datfile)
        datfilepath = Path(os.path.normpath(datfilepath))
        # Load LFP file 
        DataRec = np.fromfile(datfilepath, dtype="int16")
        print(f'.dat file found = {datfilepath}')
        Metadatafilepath = Path(os.path.join(datfilepath.parent.parent.parent, f'structure.oebin'))
        with open(Metadatafilepath) as f:
            metadata = json.load(f)
        samplerate = int(metadata['continuous'][0]['sample_rate'])
        numchannel = metadata['continuous'][0]['num_channels'] 
        rec_ch_list_origin = np.array([int(''.join(c for c in metadata['continuous'][0]['channels'][x]['channel_name'] if c.isdigit()))-1 for x in range(0, len(metadata['continuous'][0]['channels']))])
        rec_ch_list = np.array([int(''.join(filter(str.isdigit, c['channel_name']))) - 1  for c in metadata['continuous'][0]['channels'] if c['channel_name'].startswith('CH')])
        DataRec = DataRec.reshape(-1,numchannel)
        numchannel = len(rec_ch_list)
        DataRec = DataRec[:,:numchannel]

        # Load LFPs timestamps 
        for file_pathTS in datfilepath.parent.parent.parent.glob('**/continuous/*/timeStamps.npy'):
            print('LFPs timestamps file found')
            LFPtimestamps = np.load(file_pathTS) 
        LFPs_df=pd.DataFrame(DataRec, columns=rec_ch_list) 

        # Identify mouse
        mice = []
        pos_mice = []
        for mice_name in all_LFPcoordinates.index:
            if mice_name in datfilepath.__str__():
                mice.append(mice_name)
                pos_mice.append(datfilepath.__str__().find(mice_name)) 
        mice = [x for _, x in sorted(zip(pos_mice, mice))] # sort mouse in the same order as they appear in the path

        print(mice)
        
        for ID, mouse in enumerate(mice):

            print(mouse)            
            mouse_age = datfilepath.parents[6].name.replace(' ', '')
            print(mouse_age)            

            if mouse == 'TraitViolet' and mouse_age =='4mois':
                print('Flipping the headstage !!!! ')
                LFPs_df = flip_headstage(LFPs_df=LFPs_df, ID=ID)
                rec_ch_list =LFPs_df.keys().tolist()
                
            all_LFPcoordinates = all_LFPcoordinates.astype(str)
            for region in all_LFPcoordinates.loc[mouse].index:
                locals()[region] = []
                locals()[f'{region}_ch'] = []

            RecordedArea = []
            ChoosenChannels = []
            combined = []

            rec_ch_list_mouse = [value for value in rec_ch_list if 0+(ID*32) <= value <= 31+(ID*32)]
            for rec_ch in rec_ch_list_mouse:
                for idx, LFPcoord_str in enumerate(all_LFPcoordinates.loc[mouse]):
                    region = all_LFPcoordinates.loc[mouse].index[idx]
                    if LFPcoord_str != 'nan':
                        LFPcoord = LFPcoord_str.split('_')[:2] # only take into account the 2 first of electrode of that region                 
                        num_ch = np.where(str(rec_ch-(ID*32)) == np.array(LFPcoord))[0]
                        if len(num_ch) > 0:
                            region = all_LFPcoordinates.loc[mouse].index[idx]
                            LFP = locals()[region]
                            LFP = LFP-np.array(LFPs_df[(rec_ch)]) if len(LFP) > 0 else np.array(LFPs_df[(rec_ch)])
                            locals()[region] = LFP
                            locals()[f'{region}_ch'].append(rec_ch)
                            break
                        continue

            for region in all_LFPcoordinates.loc[mouse].index:
                LFP = locals()[region]
                LFP_ch = locals()[f'{region}_ch']
                if len(LFP) > 0:
                    combined = zscore(LFP[:,np.newaxis]) if len(combined) == 0 else np.append(combined, zscore(LFP[:,np.newaxis]), axis=1)
                    RecordedArea.append(region) 
                    ChoosenChannels.append(LFP_ch) 


            print(RecordedArea)
            print(ChoosenChannels) 



            ###### Save .pkl files ######

            LFPs_df_mouse=LFPs_df[rec_ch_list_mouse]
            LFPs_df_mouse = LFPs_df_mouse.copy()
            LFPs_df_mouse.rename(columns={x: x - (ID * 32) for x in rec_ch_list_mouse}, inplace=True)
            #LFPs_df.to_pickle(f'{LFPfile.parent}/DataFrame_rawdataDS.pkl')
            directory = f'//10.69.168.1/crnldata/forgetting/Carla/SleepRecordings/{mouse}/{mouse_age}/'
            file_name = 'DataFrame_rawdataDS.pkl'
            file_path = os.path.join(directory, file_name)
            os.makedirs(directory, exist_ok=True)    
            LFPs_df_mouse.to_pickle(file_path)


            ###### Define Wake episode with EMG ######
            if len(EMG) == 0:
                if not len(S1) == 0:
                    print('EMG from S1')
                    EMG = S1
                elif not len(oPFC) == 0:
                    print('EMG from oPFC')
                    EMG = oPFC
                elif not len(PFC) == 0:
                    print('EMG from PFC')
                    EMG = PFC

            # Filter parameter :
            f_lowcut = 200.
            f_hicut = 400.
            N = 4
            fs = 1000
            nyq = 0.5 * fs
            Wn = [f_lowcut/nyq,f_hicut/nyq]  # Nyquist frequency fraction

            # Filter creation :
            b, a = signal.butter(N, Wn, 'band')
            filt_EMG = signal.filtfilt(b, a, EMG)

            # Downsample EMG
            filt_EMG = signal.filtfilt(b, a, EMG)
            newsamplingrate = 100

            # Downsample EMG
            filt_EMG_DS = filt_EMG[::int(samplerate/newsamplingrate)]
            m_EMG = np.mean(filt_EMG_DS)
            sd_EMG = np.std(filt_EMG_DS)

            LowFactorSd=2
            HighFactorSd=3

            # Assigning values wake (1, 2) and sleep (0)
            numpnts = filt_EMG_DS.size
            EMGstatusRaw = np.zeros(numpnts)
            for ind in range(numpnts):
                if filt_EMG_DS[ind]<(m_EMG + LowFactorSd*sd_EMG): 
                    EMGstatusRaw[ind] = 0
                elif filt_EMG_DS[ind]>(m_EMG + HighFactorSd*sd_EMG):
                    EMGstatusRaw[ind] = 2
                else:
                    EMGstatusRaw[ind] = 1

            # Expanding borders for wake (1, 2) and sleep (0) to ±1 s around detected muscular activity
            EMGstatusRaw2 = np.zeros(numpnts)
            for ind in range(numpnts):
                if EMGstatusRaw[ind]>1:
                    EMGstatusRaw2[int(ind-newsamplingrate):int(ind+newsamplingrate)] = 2
                elif EMGstatusRaw[ind]==1:
                    for ind2 in range(int(ind-newsamplingrate), int(ind+newsamplingrate)):
                        if ind2==numpnts:
                            break
                        elif EMGstatusRaw2[ind2]<2:
                            EMGstatusRaw2[ind2] = 1

            EMGStatusBoolLib = (EMGstatusRaw2>1)
            EMGStatusBoolCons = (EMGstatusRaw2>0)

            EMG_Status_DS = EMGStatusBoolCons[::newsamplingrate]

            EMG_scoring = pd.DataFrame({
                "time": range(len(EMG_Status_DS)),
                "duration": 1,
                "label": ["Wake" if x else "Sleep" for x in EMG_Status_DS]
            })

            EMG_scoring_filename = os.path.join(Path(file_path).parent, f'EMG_scoring.csv')
            EMG_scoring.to_csv(EMG_scoring_filename, index=False)


            # Find the start of the rec 
            for root, dirs, files in os.walk(Path(f'{datfilepath.parents[4]}')):
                xmlfiles = [f for f in files if f.endswith(('.xml'))]
                for xmlfile in xmlfiles:
                    xmlfilepath = os.path.join(root, xmlfile)
                    xmlfilepath = Path(os.path.normpath(xmlfilepath))
                    tree = ET.parse(xmlfilepath)
                    settings = tree.getroot()
            for infos in settings.findall('INFO'):
                date = infos.find('DATE').text
                start_hour=date.split(' ')[-1]
                            
            HaldDay, delay= closest_checkpoint(start_hour)


            df_episodes = regrouper_labels_consecutifs(EMG_scoring)

            dureePeriod_en_min = df_episodes['duration'].sum()/ 60
            duree_moyenne_en_min = {}
            duree_total_en_min = {}
            nombre_episodes = {}
            for label in df_episodes['label'].unique():
                episodes_label = df_episodes[df_episodes['label'] == label]
                duree_moyenne_en_min[label]  = episodes_label['duration'].mean()/ 60
                duree_total_en_min[label]  = episodes_label['duration'].sum()/ 60
                nombre_episodes[label]  = len(episodes_label)

            df_Wakeepisodes= df_episodes[df_episodes['label'] == 'Wake']
            df_MicroArousal= df_Wakeepisodes[df_Wakeepisodes['duration'] < 5]
            df_Wake_NoMicroArousal= df_Wakeepisodes[df_Wakeepisodes['duration'] >= 5]

            # Store results in table

            Summary_table2.loc[counter2, 'Mouse_ID'] = all_MouseID.loc[mouse]['MouseID']
            Summary_table2.loc[counter2, 'ImplantColorCode'] = mouse
            Summary_table2.loc[counter2, 'Genotype'] = all_MouseID.loc[mouse]['Genotype']
            Summary_table2.loc[counter2, 'Age'] = mouse_age
            Summary_table2.loc[counter2, 'Session'] = ' '.join([str(x) for x in date.split(' ')[:-1]])
            Summary_table2.loc[counter2, 'Session_time'] = start_hour
            Summary_table2.loc[counter2, 'Sex'] = all_MouseID.loc[mouse]['Sex']
            Summary_table2.loc[counter2, 'Duree_Period_min'] = round(dureePeriod_en_min,2)
            Summary_table2.loc[counter2, 'Nb_Total_episodes'] =int(len(df_episodes))
            Summary_table2.loc[counter2, 'Nb_Episod_Wake'] = int(nombre_episodes['Wake']) if 'Wake' in nombre_episodes else 0
            Summary_table2.loc[counter2, 'Nb_Episod_Sleep'] = int(nombre_episodes['Sleep']) if 'Sleep' in nombre_episodes else 0            
            Summary_table2.loc[counter2, 'Nb_Episod_RealWake'] = int(len(df_Wake_NoMicroArousal))
            Summary_table2.loc[counter2, 'Nb_MicroArousal'] = int(len(df_MicroArousal))            
            Summary_table2.loc[counter2, 'Nb_MicroArousal_per_h'] = int(len(df_MicroArousal)/(dureePeriod_en_min/60))
            Summary_table2.loc[counter2, 'DurTot_Wake_min'] = round(duree_total_en_min['Wake'],2) if 'Wake' in duree_total_en_min else 0
            Summary_table2.loc[counter2, 'DurTot_Sleep_min'] = round(duree_total_en_min['Sleep'],2) if 'Sleep' in duree_total_en_min else 0
            Summary_table2.loc[counter2, '%Tot_Wake'] = round(duree_total_en_min['Wake']/dureePeriod_en_min*100,2) if 'Wake' in duree_total_en_min else 0
            Summary_table2.loc[counter2, '%Tot_Sleep'] = round(duree_total_en_min['Sleep']/dureePeriod_en_min*100,2) if 'Sleep' in duree_total_en_min else 0
            Summary_table2.loc[counter2, 'Avg_durWake_sec'] = round(duree_moyenne_en_min['Wake']*60,2)  if 'Wake' in duree_total_en_min else 0
            Summary_table2.loc[counter2, 'Avg_durRealWake_sec'] = round(df_Wake_NoMicroArousal['duration'].mean(),2)
            Summary_table2.loc[counter2, 'Avg_durSleep_sec'] = round(duree_moyenne_en_min['Sleep']*60,2) if 'Sleep' in duree_total_en_min else 0

            counter2 +=1


            duration = 12 * 60 * 60  # 43200 seconds
            data = []
            if delay < len(EMG_scoring):
                data.append({"start": int(0), "end": int(delay)})
                e = delay
                for i in range(int((len(EMG_scoring)-delay)/60/60 // 12)):
                    s = delay + i * duration
                    e = s + duration
                    data.append({"start": int(s), "end": int(e)})
                data.append({"start": int(e), "end": int(len(EMG_scoring))})
            else:
                data.append({"start": int(0), "end": int(len(EMG_scoring))})
            periods= pd.DataFrame(data)

            counterDayPeriod = 0
            coutnerNightPeriod = 0

            for period_nb, (start, end) in enumerate(zip(periods['start'], periods['end'])):

                df= EMG_scoring.iloc[start:end]
                df_episodes = regrouper_labels_consecutifs(df)

                dureePeriod_en_min = df_episodes['duration'].sum()/ 60
                duree_moyenne_en_min = {}
                duree_total_en_min = {}
                nombre_episodes = {}
                for label in df_episodes['label'].unique():
                    episodes_label = df_episodes[df_episodes['label'] == label]
                    duree_moyenne_en_min[label]  = episodes_label['duration'].mean()/ 60
                    duree_total_en_min[label]  = episodes_label['duration'].sum()/ 60
                    nombre_episodes[label]  = len(episodes_label)

                
                df_Wakeepisodes = df_episodes[df_episodes['label'] == 'Wake']
                df_MicroArousal = df_Wakeepisodes[(df_Wakeepisodes['duration'] >= 3) & (df_Wakeepisodes['duration'] <= 15)]
                df_Wake_NoMicroArousal = df_Wakeepisodes[(df_Wakeepisodes['duration'] < 3) | (df_Wakeepisodes['duration'] > 15)]

                counterDayPeriod +=1 if HaldDay == 'Day' else  0
                coutnerNightPeriod +=1 if HaldDay == 'Night' else  0

                # Store results in table
                Summary_table.loc[counter, 'Mouse_ID'] = all_MouseID.loc[mouse]['MouseID']
                Summary_table.loc[counter, 'ImplantColorCode'] = mouse
                Summary_table.loc[counter, 'Genotype'] = all_MouseID.loc[mouse]['Genotype']
                Summary_table.loc[counter, 'Age'] = mouse_age
                Summary_table.loc[counter, 'Session'] = ' '.join([str(x) for x in date.split(' ')[:-1]])
                Summary_table.loc[counter, 'Session_time'] = start_hour
                Summary_table.loc[counter, 'Sex'] = all_MouseID.loc[mouse]['Sex']
                Summary_table.loc[counter, 'Period_Type'] = HaldDay
                Summary_table.loc[counter, 'Period_Type_Nb'] = counterDayPeriod if HaldDay == 'Day' else coutnerNightPeriod
                Summary_table.loc[counter, 'Period_Nb'] = period_nb

                Summary_table.loc[counter, 'Duree_Period_min'] = round(dureePeriod_en_min,2)

                Summary_table.loc[counter, 'Nb_Total_episodes'] =int(len(df_episodes))
                Summary_table.loc[counter, 'Nb_Episod_Wake'] = int(nombre_episodes['Wake']) if 'Wake' in nombre_episodes else 0
                Summary_table.loc[counter, 'Nb_Episod_Sleep'] = int(nombre_episodes['Sleep']) if 'Sleep' in nombre_episodes else 0
                
                Summary_table.loc[counter, 'Nb_Episod_RealWake'] = int(len(df_Wake_NoMicroArousal))
                Summary_table.loc[counter, 'Nb_MicroArousal'] = int(len(df_MicroArousal))
                
                Summary_table.loc[counter, 'Nb_MicroArousal_per_h'] = int(len(df_MicroArousal)/(dureePeriod_en_min/60))

                Summary_table.loc[counter, 'DurTot_Wake_min'] = round(duree_total_en_min['Wake'],2) if 'Wake' in duree_total_en_min else 0
                Summary_table.loc[counter, 'DurTot_Sleep_min'] = round(duree_total_en_min['Sleep'],2) if 'Sleep' in duree_total_en_min else 0

                Summary_table.loc[counter, '%Tot_Wake'] = round(duree_total_en_min['Wake']/dureePeriod_en_min*100,2) if 'Wake' in duree_total_en_min else 0
                Summary_table.loc[counter, '%Tot_Sleep'] = round(duree_total_en_min['Sleep']/dureePeriod_en_min*100,2) if 'Sleep' in duree_total_en_min else 0

                Summary_table.loc[counter, 'Avg_durWake_sec'] = round(duree_moyenne_en_min['Wake']*60,2)  if 'Wake' in duree_total_en_min else 0
                Summary_table.loc[counter, 'Avg_durRealWake_sec'] = round(df_Wake_NoMicroArousal['duration'].mean(),2)
                Summary_table.loc[counter, 'Avg_durSleep_sec'] = round(duree_moyenne_en_min['Sleep']*60,2) if 'Sleep' in duree_total_en_min else 0

                HaldDay = 'Night' if HaldDay == 'Day' else 'Day'
                
                counter+=1

            if not mouse in EMGHypno_dict:
                EMGHypno_dict[mouse]={}

            EMGHypno_dict[mouse][mouse_age]={}
            EMGHypno_dict[mouse][mouse_age]['filtEMG'] = filt_EMG_DS[::10]
            EMGHypno_dict[mouse][mouse_age]['Hypno']= EMGStatusBoolCons[::10]
            EMGHypno_dict[mouse][mouse_age]['StartTime'] = start_hour

            # Create small plot of Hypno & EMG

            genotype=Summary_table[Summary_table['ImplantColorCode']==mouse]['Genotype'].unique().tolist()[0]
            sex=Summary_table[Summary_table['ImplantColorCode']==mouse]['Sex'].unique().tolist()[0]
            fig, ax = plt.subplots(figsize=(15,2))
            EMGfilt_DS = zscore(EMGHypno_dict[mouse][mouse_age]['filtEMG'])
            starttime = EMGHypno_dict[mouse][mouse_age]['StartTime']
            Hypno= EMGHypno_dict[mouse][mouse_age]['Hypno']
            start = pd.to_datetime(starttime)
            fs = 10          # 100 Hz or 10 HZ
            N = len(EMGfilt_DS)
            dt = 1.0 / fs     # sample spacing in seconds
            offsets = np.arange(N) * dt
            datetimes = start + pd.to_timedelta(offsets, unit='s')
            y = EMGfilt_DS
            unique_days = sorted({d.date() for d in datetimes})
            ticks = []
            for d in unique_days:
                ticks.append(pd.to_datetime(f"{d} 07:00:00"))
                ticks.append(pd.to_datetime(f"{d} 19:00:00"))
            ticks = [t for t in ticks if datetimes.min() <= t <= datetimes.max()]
            ax.plot(datetimes, y, linewidth=0.5)
            ax.set_ylabel("EMG & Hypno")
            ax.set_xticks(sorted(ticks))
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
            ax.plot(datetimes, Hypno*10-50, linewidth=0.5)
            plt.xticks(rotation=45)
            ax.set_xlim(datetimes.min(), datetimes.max())
            ax.set_title(f'{mouse_age}', fontsize=10)
            plt.tight_layout()
            fig.suptitle(f'{genotype} {sex} {mouse}', fontsize=10)
            plt.tight_layout()
            plt.savefig(f'//10.69.168.1/crnldata/forgetting/Carla/SleepRecordings/{mouse}/{mouse_age}/EMGHypno_{mouse}_{mouse_age}.png', dpi=300, bbox_inches='tight', facecolor='white')

            # Save tables & dict

            filenameOut = f'//10.69.168.1/crnldata/forgetting/Carla/SleepRecordings/SummaryPeriod_table.xlsx'
            with pd.ExcelWriter(filenameOut) as writer:
                Summary_table.to_excel(writer, index=False)

            filenameOut = f'//10.69.168.1/crnldata/forgetting/Carla/SleepRecordings/Summary_table.xlsx'
            with pd.ExcelWriter(filenameOut) as writer:
                Summary_table2.to_excel(writer, index=False)

            with open(f'//10.69.168.1/crnldata/forgetting/Carla/SleepRecordings/EMGHypno_dict.pkl', 'wb') as f:
                pickle.dump(EMGHypno_dict, f)

print(f"Analyse terminée ! Tableau sauvegardé dans : {filenameOut}")
print(f"Nombre total d'essais analysés : {counter}") 


Starts...
.dat file found = \\10.69.168.1\crnldata\forgetting\Carla\OpenEphysData\Sleep\10 mois\RondRouge_RondOrange_RondJaune_2025-11-03_17-17-15_Sleep_10mois\Record Node 105\experiment1\recording1\continuous\OE_FPGA_Acquisition_Board-106.Rhythm Data\continuous.dat
LFPs timestamps file found
['RondRouge', 'RondOrange', 'RondJaune']
RondRouge
10mois
['EMG', 'PFC', 'CA1', 'oPFC', 'EnthC', 'RH']
[[18], [20, 21], [15], [17], [22], [12]]
RondOrange
10mois
['EMG', 'S1', 'PFC', 'CA1', 'oPFC', 'EnthC', 'RH']
[[42], [50, 51], [54, 55], [46, 47], [52], [44], [48, 49]]
RondJaune
10mois
['EMG', 'S1', 'PFC', 'CA1', 'oPFC', 'EnthC', 'RH']
[[74], [83], [84, 85], [78, 79], [86, 87], [76], [80, 81]]
.dat file found = \\10.69.168.1\crnldata\forgetting\Carla\OpenEphysData\Sleep\10 mois\RondViolet_RondVert_RondBleu_RondMarron_2025-10-29_17-37-35_Sleep_10mois\Record Node 105\experiment1\recording1\continuous\OE_FPGA_Acquisition_Board-106.Rhythm Data\continuous.dat
LFPs timestamps file found
['RondViolet',

C:\Users\AudreyHay\AppData\Local\Temp\ipykernel_18524\111146319.py:343: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(figsize=(15,2))


RondJaune
11mois
['EMG', 'S1', 'PFC', 'CA1', 'oPFC', 'EnthC', 'RH']
[[74], [83], [84, 85], [78, 79], [86, 87], [76], [80, 81]]
.dat file found = \\10.69.168.1\crnldata\forgetting\Carla\OpenEphysData\Sleep\11 mois\RondViolet_RondVert_RondBleu_RondMarron_2025-12-01_15-25-02_Sleep_11mois\Record Node 105\experiment2\recording1\continuous\OE_FPGA_Acquisition_Board-106.Rhythm Data\continuous.dat
LFPs timestamps file found
['RondViolet', 'RondVert', 'RondBleu', 'RondMarron']
RondViolet
11mois
['EMG', 'S1', 'PFC', 'CA1', 'oPFC', 'EnthC', 'RH']
[[10], [14, 15], [16, 17], [22, 23], [12, 13], [20, 21], [18, 19]]
RondVert
11mois
['EMG', 'S1', 'PFC', 'CA1', 'oPFC', 'EnthC', 'RH']
[[42], [50], [52, 53], [46, 47], [54, 55], [45], [48, 49]]
RondBleu
11mois
['EMG', 'S1', 'PFC', 'CA1', 'oPFC', 'EnthC', 'RH']
[[74], [78, 79], [80, 81], [82, 83], [76, 77], [84, 85], [86, 87]]
RondMarron
11mois
['EMG', 'S1', 'PFC', 'CA1', 'oPFC', 'EnthC', 'RH']
[[106], [114, 115], [116, 117], [110, 111], [118, 119], [108, 

In [1]:

import pandas as pd 

summary = pd.read_excel("//10.69.168.1/crnldata/forgetting/Carla/SleepRecordings/Summary_table.xlsx")

table = summary.pivot_table(values=['DurTot_Sleep_min',
'Duree_Period_min', 'Nb_MicroArousal', 'Nb_Episod_Sleep'], index=['Genotype',
'Age','Mouse_ID','ImplantColorCode'], aggfunc={'DurTot_Sleep_min':
'sum', 'Duree_Period_min': 'sum', 'Nb_MicroArousal':
'sum', 'Nb_Episod_Sleep': 'sum'}).round(2)

table['%SleepTot']=round(table['DurTot_Sleep_min']/table['Duree_Period_min']*100,2)

table['Avg_DurSleepEpisode']=round(table['DurTot_Sleep_min']/table['Nb_Episod_Sleep']*100,2)

table['MicroArousal_per_h']=round(table['Nb_MicroArousal']/(table['Duree_Period_min']/60),2)

table

DurTot_Sleep_min  \
Genotype Age    Mouse_ID   ImplantColorCode                     
APPPS1   10mois AHAD01.37  RondRouge                  1411.22   
                AHAD01.38  RondOrange                 1566.95   
                AHAD01.40  RondViolet                 2104.62   
                AHAD02.01  TraitRouge                 1548.28   
                AHAD02.03  TraitViolet                1592.92   
                AHAD02.05  TraitVert                  2755.13   
                AHAD02.06  TraitOrange                2257.83   
                AHAD11.69  TriangleBleu               1388.67   
                AHAD11.71  TriangleRouge              1495.69   
         11mois AHAD01.37  RondRouge                  1321.85   
                AHAD01.38  RondOrange                 1790.20   
                AHAD01.40  RondViolet                 1886.25   
                AHAD02.01  TraitRouge                 1536.18   
                AHAD02.03  TraitViolet                2375.83   
                AHAD02.05  TraitVert                  2536.03   
                AHAD02.06  TraitOrange                2310.98   
                AHAD11.69  TriangleBleu               1295.27   
                AHAD11.71  TriangleRouge              2015.57   
         12mois AHAD01.37  RondRouge                  1698.83   
                AHAD01.38  RondOrange                 1516.68   
                AHAD02.01  TraitRouge                 1767.22   
                AHAD02.03  TraitViolet                2730.05   
                AHAD02.05  TraitVert                  2422.60   
                AHAD02.06  TraitOrange                2328.73   
         3mois  AHAD01.40  RondViolet                 2029.60   
                AHAD11.101 CarreRouge                 1642.93   
                AHAD12.104 CarreNoir                  1556.07   
                AHAD12.106 CarreOrange                1440.55   
                AHAD12.109 CarreJaune                 1486.03   
WT       10mois AHAD01.39  RondJaune                  1892.22   
                AHAD01.41  RondVert                   1461.87   
                AHAD01.42  RondBleu                   1612.38   
                AHAD01.43  RondMarron                 2021.28   
                AHAD11.60  TriangleBlanc              1883.55   
                AHAD11.66  TriangleViolet             1711.72   
                AHAD11.70  TriangleMarron             1650.25   
         11mois AHAD01.39  RondJaune                  2082.87   
                AHAD01.41  RondVert                   1515.37   
                AHAD01.42  RondBleu                   1931.47   
                AHAD01.43  RondMarron                 1860.30   
                AHAD11.60  TriangleBlanc              1863.38   
                AHAD11.66  TriangleViolet             1228.80   
                AHAD11.70  TriangleMarron             1788.20   
         12mois AHAD01.39  RondJaune                  2332.95   
                AHAD01.41  RondVert                   1799.92   
                AHAD01.42  RondBleu                   1834.63   
                AHAD01.43  RondMarron                 1659.88   
         3mois  AHAD01.39  RondJaune                  2572.23   
                AHAD01.41  RondVert                   1496.73   
                AHAD01.42  RondBleu                   1456.23   
                AHAD11.103 CarreMarron                1654.30   
                AHAD12.105 CarreBlanc                 1545.12   
                AHAD12.107 CarreVert                  1503.00   
                AHAD12.108 CarreViolet                1423.30   

                                             Duree_Period_min  \
Genotype Age    Mouse_ID   ImplantColorCode                     
APPPS1   10mois AHAD01.37  RondRouge                  2928.48   
                AHAD01.38  RondOrange                 2928.48   
                AHAD01.40  RondViolet                 2904.35   
                AHAD02.01  TraitRouge                 2948.23   
              